In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Customer Review Analyzer — Sentiment, Topics & Pain Points

## 1. Project Overview

**Task:** Analyze a dataset of customer reviews to extract sentiment, identify recurring topics/themes, surface pain points, and detect feature requests — all using a local LLM.

**Approach:** Multi-pass LLM analysis — classify sentiment → extract topics → identify pain points & feature requests → aggregate into a dashboard.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for all analysis
- `LangChain` — prompt templates
- Pure Python — no pandas needed; we keep it lightweight

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Classify sentiment** (positive / neutral / negative) with confidence scores |
| 2 | **Extract topics** from unstructured review text |
| 3 | **Identify pain points** — what frustrates customers? |
| 4 | **Detect feature requests** — what do customers want built? |
| 5 | **Cluster themes** — group reviews by recurring patterns |
| 6 | **Build a summary dashboard** — aggregate findings into actionable insights |
| 7 | **Structured JSON output** — get reliable structured data from LLM responses |

## 3. Problem Statement

Companies receive thousands of customer reviews across app stores, support tickets, and social media. Reading every review is impossible. Product managers need:

- **Sentiment breakdown** — are reviews trending positive or negative?
- **Topic distribution** — what are customers talking about?
- **Pain points** — what specific problems do customers report?
- **Feature requests** — what are customers asking for?

## 4. Why This Project Matters

- **Voice of the Customer (VoC)** analysis is a core product management skill
- Shows how to use LLMs for **multi-label classification** (one review → many labels)
- **Structured extraction** from free-text is the #1 enterprise LLM use case
- Combines classification + extraction + summarization in one pipeline
- Every SaaS company does this: Zendesk, Intercom, and ProductBoard all offer AI review analysis

## 5. Pipeline Architecture

```
Customer Reviews (text dataset)
        │
        ▼
  [Pass 1: Sentiment] ── positive / neutral / negative + confidence
        │
        ▼
  [Pass 2: Topic Extraction] ── 1-3 topics per review
        │
        ▼
  [Pass 3: Pain Points & Feature Requests] ── structured extraction
        │
        ▼
  [Aggregate] ── count sentiments, rank topics, group pain points
        │
        ▼
  [Theme Clustering] ── LLM groups similar pain points into themes
        │
        ▼
  [Dashboard] ── summary tables, top issues, recommendations
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core

print("Dependencies: langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"

TEMP_CLASSIFY = 0.0   # Sentiment — deterministic
TEMP_EXTRACT = 0.1    # Topic/pain point extraction — mostly factual
TEMP_SUMMARY = 0.3    # Summary generation — slight creativity

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL}")
print(f"  Temperatures: classify={TEMP_CLASSIFY}, extract={TEMP_EXTRACT}, "
      f"summary={TEMP_SUMMARY}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    # Find first { or [
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.0) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Review Dataset

## 10. Customer Review Dataset

We use a realistic dataset of 20 product reviews for a fictional project-management SaaS tool. The reviews span a range of sentiments, topics, and customer types — providing enough variety for meaningful analysis.

Each review includes:
- **author** — the reviewer's name
- **rating** — 1-5 stars
- **text** — the review body
- **source** — where the review came from

In [ ]:
REVIEWS = [
    {
        "id": 1, "author": "Sarah M.", "rating": 5, "source": "App Store",
        "text": "Absolutely love this tool! The Kanban boards are so intuitive and the "
                "mobile app syncs perfectly. My team of 8 has been using it for 6 months "
                "and productivity is up 30%. The Gantt chart view is a game changer for "
                "planning sprints. Worth every penny of the Professional plan."
    },
    {
        "id": 2, "author": "Mike T.", "rating": 2, "source": "G2",
        "text": "The app crashes constantly on Android. I've reported it three times and "
                "support just tells me to clear the cache. It's been two months and still "
                "no fix. The desktop version is fine but I need reliable mobile access "
                "when I'm on site visits. Considering switching to Asana."
    },
    {
        "id": 3, "author": "Lisa K.", "rating": 4, "source": "Capterra",
        "text": "Great for small teams. The pricing is fair and onboarding was smooth. "
                "Only complaint: the reporting dashboard is too basic. I need custom date "
                "ranges and the ability to export reports as PDF. Currently I have to "
                "screenshot everything which feels unprofessional."
    },
    {
        "id": 4, "author": "James R.", "rating": 1, "source": "App Store",
        "text": "Terrible customer support. I was charged for 12 users when I only have 5. "
                "Took 3 weeks to get a response and they refused to refund the difference. "
                "The billing system is confusing and there's no way to easily see what "
                "you're being charged for. Avoid if you value your money."
    },
    {
        "id": 5, "author": "Priya S.", "rating": 5, "source": "G2",
        "text": "The Slack integration is flawless. Every task update triggers a "
                "notification in our channel and the team loves it. We tried Monday.com "
                "and ClickUp before this — both had buggy integrations. This one just works. "
                "The API is well documented too, we built a custom dashboard on top of it."
    },
    {
        "id": 6, "author": "David L.", "rating": 3, "source": "Trustpilot",
        "text": "Middle of the road. It does what it promises but nothing exciting. "
                "The UI looks dated compared to Notion or Linear. I wish they would "
                "modernize the interface — dark mode would be a great start. Also, "
                "loading times are slow when you have 500+ tasks in a project."
    },
    {
        "id": 7, "author": "Emma W.", "rating": 4, "source": "App Store",
        "text": "I manage a remote team of 15 and this tool keeps everyone aligned. "
                "The time tracking feature is accurate and the weekly summary emails "
                "are genuinely useful. My only wish is for a calendar view that integrates "
                "with Google Calendar. Right now I have to manually sync deadlines."
    },
    {
        "id": 8, "author": "Carlos F.", "rating": 1, "source": "G2",
        "text": "We signed up for the Enterprise plan expecting SSO support. After "
                "paying for 3 months we found out SAML SSO is 'coming soon' and only "
                "basic SSO via Google is available. This was not clear on the pricing "
                "page. We need SAML for compliance. Very misleading."
    },
    {
        "id": 9, "author": "Nina P.", "rating": 5, "source": "Capterra",
        "text": "Switched from Trello and never looked back. The automation workflows "
                "save us hours every week — auto-assigning tasks based on labels, "
                "moving cards when status changes, sending Slack pings. The learning "
                "curve is minimal and customer support was incredibly helpful during setup."
    },
    {
        "id": 10, "author": "Tom H.", "rating": 2, "source": "Trustpilot",
        "text": "The search function is broken. I have 200 tasks and searching by keyword "
                "returns irrelevant results or nothing at all. Also, there's no way to "
                "filter by multiple labels at once. Basic functionality that should have "
                "been built from day one. Getting frustrating."
    },
    {
        "id": 11, "author": "Aisha J.", "rating": 4, "source": "G2",
        "text": "The CRM module was a surprise — didn't expect it to be this good in "
                "a project management tool. We track deals and tasks in one place now. "
                "Would love to see email tracking added to the CRM. Currently we use "
                "HubSpot for that which is redundant."
    },
    {
        "id": 12, "author": "Robert D.", "rating": 3, "source": "App Store",
        "text": "The free tier is surprisingly generous but upgrading to Professional "
                "feels steep at $29/user. For a team of 20 that's $580/month which is "
                "hard to justify to management. Would love a mid-tier plan around $15-20 "
                "that includes basic reporting but not the full CRM."
    },
    {
        "id": 13, "author": "Yuki T.", "rating": 5, "source": "Capterra",
        "text": "Onboarding experience was the best I've seen in any SaaS tool. "
                "Interactive tutorials, template projects, and a dedicated CSM who "
                "helped us migrate from Jira. Our team was productive on day one. "
                "The import tool handled 3,000 Jira tickets without issues."
    },
    {
        "id": 14, "author": "Grace C.", "rating": 2, "source": "Trustpilot",
        "text": "Notifications are overwhelming. I get pinged for every tiny change — "
                "someone edits a description, I get notified. Someone changes a due date, "
                "notified. There's no granular control over notification types. "
                "I've resorted to muting the app which defeats the purpose."
    },
    {
        "id": 15, "author": "Wei Z.", "rating": 4, "source": "G2",
        "text": "The API rate limits are generous on the Enterprise plan (1000 req/min). "
                "We built a custom reporting layer for our VP of Engineering that pulls "
                "data every 5 minutes. Documentation is excellent with working code "
                "examples. Wish they had GraphQL support though."
    },
    {
        "id": 16, "author": "Alex B.", "rating": 1, "source": "App Store",
        "text": "Lost 2 weeks of data after a sync error between mobile and desktop. "
                "Support said it was a 'known issue' and offered no recovery option. "
                "For a paid product this is unacceptable. We had to manually re-enter "
                "everything. No backup or version history feature exists."
    },
    {
        "id": 17, "author": "Maria G.", "rating": 4, "source": "Capterra",
        "text": "Invoicing feature saved us from using a separate tool. We send "
                "invoices directly from project tasks and track payments in one place. "
                "The Stripe integration works perfectly. One improvement: add support "
                "for multiple currencies and tax region settings."
    },
    {
        "id": 18, "author": "Kevin N.", "rating": 3, "source": "G2",
        "text": "Works well for basic project management but the lack of dependencies "
                "between tasks is a real limitation. In complex projects I need to set "
                "task A as a blocker for task B. Currently there's no way to do this. "
                "Also missing: recurring tasks and task templates."
    },
    {
        "id": 19, "author": "Sana A.", "rating": 5, "source": "Trustpilot",
        "text": "The best value for money in project management tools. Our nonprofit "
                "got the 50% educational discount and it's incredible. We manage 12 "
                "grant projects and 30 volunteers through this tool. The permission "
                "system lets us give volunteers limited access without exposing financials."
    },
    {
        "id": 20, "author": "Jake P.", "rating": 2, "source": "App Store",
        "text": "The offline mode is basically useless. It says 'offline mode available' "
                "but you can only VIEW tasks, not edit or create new ones. I work in "
                "areas with spotty WiFi and this makes the mobile app unreliable. "
                "Competitors like Todoist handle offline editing much better."
    },
]

print(f"Dataset loaded: {len(REVIEWS)} reviews")
rating_dist = Counter(r["rating"] for r in REVIEWS)
for star in sorted(rating_dist):
    bar = "★" * star + "☆" * (5 - star)
    print(f"  {bar}  {rating_dist[star]} reviews")
avg = sum(r["rating"] for r in REVIEWS) / len(REVIEWS)
print(f"  Average rating: {avg:.1f}/5")

## 11. Data Inspection

In [ ]:
print("REVIEW DATASET — PREVIEW")
print("=" * 70)

for r in REVIEWS[:5]:
    stars = "★" * r["rating"] + "☆" * (5 - r["rating"])
    print(f"\n[#{r['id']}] {r['author']} — {stars} ({r['source']})")
    print(textwrap.fill(r["text"], width=70))

print(f"\n... and {len(REVIEWS) - 5} more reviews")

---

# Part B — Sentiment Analysis

## 12. Sentiment Classification

For each review, classify as **positive**, **neutral**, or **negative** and assign a confidence score. We also extract a one-line summary of the reviewer's main sentiment.

In [ ]:
SENTIMENT_SYSTEM = """You are a sentiment analysis expert. Classify customer reviews accurately.
Consider the overall tone, not just individual words. A review with a complaint but
overall positive tone should be classified as positive."""

SENTIMENT_PROMPT = """Analyze the sentiment of this customer review.

Review (rating: {rating}/5):
"{text}"

Return ONLY JSON:
{{
  "sentiment": "positive" or "neutral" or "negative",
  "confidence": 0.0 to 1.0,
  "summary": "one sentence summary of sentiment"
}}"""

print("SENTIMENT ANALYSIS")
print("=" * 70)

sentiment_results = []

for r in REVIEWS:
    resp = ask(
        SENTIMENT_PROMPT.format(rating=r["rating"], text=r["text"]),
        system=SENTIMENT_SYSTEM,
        temperature=TEMP_CLASSIFY,
    )
    parsed = parse_json(resp)

    if parsed:
        result = {
            "id": r["id"],
            "author": r["author"],
            "rating": r["rating"],
            "sentiment": parsed.get("sentiment", "unknown"),
            "confidence": parsed.get("confidence", 0),
            "summary": parsed.get("summary", ""),
        }
    else:
        result = {
            "id": r["id"], "author": r["author"], "rating": r["rating"],
            "sentiment": "unknown", "confidence": 0, "summary": "parse error",
        }

    sentiment_results.append(result)

    icon = {"positive": "🟢", "neutral": "🟡", "negative": "🔴"}.get(
        result["sentiment"], "⚪")
    print(f"  {icon} #{r['id']:2d} {r['author']:10s} "
          f"({result['sentiment']:8s} {result['confidence']:.0%}) "
          f"— {result['summary'][:60]}")

print(f"\nProcessed {len(sentiment_results)} reviews")

## 13. Sentiment Distribution

In [ ]:
sent_counts = Counter(r["sentiment"] for r in sentiment_results)
total = len(sentiment_results)

print("SENTIMENT DISTRIBUTION")
print("=" * 50)

for label in ["positive", "neutral", "negative"]:
    count = sent_counts.get(label, 0)
    pct = 100 * count / total
    bar = "█" * int(pct / 2)
    icon = {"positive": "🟢", "neutral": "🟡", "negative": "🔴"}[label]
    print(f"  {icon} {label:8s}  {bar:25s}  {count:2d} ({pct:.0f}%)")

# Confidence analysis
avg_conf = sum(r["confidence"] for r in sentiment_results) / total
low_conf = [r for r in sentiment_results if r["confidence"] < 0.7]
print(f"\nAverage confidence: {avg_conf:.0%}")
if low_conf:
    print(f"Low-confidence reviews ({len(low_conf)}):")
    for r in low_conf:
        print(f"  #{r['id']} {r['author']} — {r['sentiment']} ({r['confidence']:.0%})")

# Sentiment vs rating alignment
print(f"\nSentiment vs Rating agreement:")
agree = 0
for r in sentiment_results:
    expected = "positive" if r["rating"] >= 4 else ("negative" if r["rating"] <= 2 else "neutral")
    match = r["sentiment"] == expected
    agree += int(match)
    if not match:
        print(f"  ⚠ #{r['id']} rated {r['rating']}★ but sentiment={r['sentiment']}")
print(f"  Agreement: {agree}/{total} ({100*agree/total:.0f}%)")

---

# Part C — Topic Extraction

## 14. Extract Topics per Review

For each review, extract 1-3 topics from a predefined taxonomy. This gives us structured data we can aggregate.

In [ ]:
TOPIC_SYSTEM = """You extract topics from customer reviews. Use ONLY topics from the
provided taxonomy. Each review should map to 1-3 topics."""

TOPIC_PROMPT = """Extract topics from this customer review.

TAXONOMY (choose only from these):
- Mobile App
- Integrations
- UI/UX Design
- Performance/Speed
- Customer Support
- Pricing/Billing
- Reporting/Analytics
- Onboarding
- Reliability/Bugs
- Features - Project Management
- Features - CRM
- Features - Invoicing
- Security/Compliance
- API/Developer
- Notifications
- Data/Backup

Review:
"{text}"

Return ONLY JSON:
{{
  "topics": ["Topic1", "Topic2"],
  "primary_topic": "TopicX"
}}"""

print("TOPIC EXTRACTION")
print("=" * 70)

topic_results = []

for r in REVIEWS:
    resp = ask(
        TOPIC_PROMPT.format(text=r["text"]),
        system=TOPIC_SYSTEM,
        temperature=TEMP_EXTRACT,
    )
    parsed = parse_json(resp)

    if parsed:
        result = {
            "id": r["id"],
            "topics": parsed.get("topics", []),
            "primary_topic": parsed.get("primary_topic", ""),
        }
    else:
        result = {"id": r["id"], "topics": [], "primary_topic": "unknown"}

    topic_results.append(result)
    print(f"  #{r['id']:2d} [{result['primary_topic']:28s}] "
          f"+ {', '.join(result['topics'][:3])}")

print(f"\nExtracted topics for {len(topic_results)} reviews")

## 15. Topic Distribution

In [ ]:
# Count all topic mentions
all_topics = []
for tr in topic_results:
    all_topics.extend(tr["topics"])

topic_counts = Counter(all_topics)

print("TOPIC DISTRIBUTION (ranked)")
print("=" * 60)

max_count = max(topic_counts.values()) if topic_counts else 1
for topic, count in topic_counts.most_common():
    bar_len = int(20 * count / max_count)
    bar = "█" * bar_len
    print(f"  {topic:30s} {bar:20s} {count}")

# Topic x Sentiment cross-tabulation
print(f"\nTOPIC × SENTIMENT")
print(f"{'Topic':30s} {'Pos':>4s} {'Neu':>4s} {'Neg':>4s}")
print("-" * 48)

topic_sentiment = {}
for tr, sr in zip(topic_results, sentiment_results):
    for topic in tr["topics"]:
        if topic not in topic_sentiment:
            topic_sentiment[topic] = {"positive": 0, "neutral": 0, "negative": 0}
        sent = sr["sentiment"]
        if sent in topic_sentiment[topic]:
            topic_sentiment[topic][sent] += 1

for topic in topic_counts.most_common():
    t = topic[0]
    if t in topic_sentiment:
        s = topic_sentiment[t]
        print(f"  {t:30s} {s['positive']:3d}  {s['neutral']:3d}  {s['negative']:3d}")

---

# Part D — Pain Points & Feature Requests

## 16. Extract Pain Points & Feature Requests

Go deeper: for each review, identify specific **pain points** (problems) and **feature requests** (wants).

In [ ]:
EXTRACT_SYSTEM = """You analyze customer reviews to extract specific pain points and
feature requests. Be precise — extract the exact issue, not vague summaries.
If a review has no pain points or feature requests, return empty arrays."""

EXTRACT_PROMPT = """Analyze this customer review and extract:

1. PAIN POINTS — specific problems, frustrations, or complaints
2. FEATURE REQUESTS — specific features or improvements the customer wants

Review:
"{text}"

Return ONLY JSON:
{{
  "pain_points": ["specific problem 1", "specific problem 2"],
  "feature_requests": ["wanted feature 1", "wanted feature 2"],
  "severity": "low" or "medium" or "high"
}}"""

print("PAIN POINT & FEATURE REQUEST EXTRACTION")
print("=" * 70)

extraction_results = []

for r in REVIEWS:
    resp = ask(
        EXTRACT_PROMPT.format(text=r["text"]),
        system=EXTRACT_SYSTEM,
        temperature=TEMP_EXTRACT,
    )
    parsed = parse_json(resp)

    if parsed:
        result = {
            "id": r["id"],
            "author": r["author"],
            "pain_points": parsed.get("pain_points", []),
            "feature_requests": parsed.get("feature_requests", []),
            "severity": parsed.get("severity", "medium"),
        }
    else:
        result = {
            "id": r["id"], "author": r["author"],
            "pain_points": [], "feature_requests": [], "severity": "unknown",
        }

    extraction_results.append(result)

    if result["pain_points"] or result["feature_requests"]:
        sev_icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(
            result["severity"], "⚪")
        print(f"\n  {sev_icon} #{r['id']} {r['author']}")
        for pp in result["pain_points"]:
            print(f"     ⚠ Pain: {pp}")
        for fr in result["feature_requests"]:
            print(f"     ✨ Want: {fr}")

total_pp = sum(len(e["pain_points"]) for e in extraction_results)
total_fr = sum(len(e["feature_requests"]) for e in extraction_results)
print(f"\nTotal pain points: {total_pp}")
print(f"Total feature requests: {total_fr}")

## 17. Rank Pain Points & Feature Requests

In [ ]:
# Collect all pain points and feature requests
all_pain_points = []
all_feature_requests = []

for e in extraction_results:
    for pp in e["pain_points"]:
        all_pain_points.append({"text": pp, "author": e["author"],
                                "severity": e["severity"], "id": e["id"]})
    for fr in e["feature_requests"]:
        all_feature_requests.append({"text": fr, "author": e["author"], "id": e["id"]})

print("TOP PAIN POINTS (by severity)")
print("=" * 70)

severity_order = {"high": 0, "medium": 1, "low": 2, "unknown": 3}
sorted_pp = sorted(all_pain_points, key=lambda x: severity_order.get(x["severity"], 3))

for i, pp in enumerate(sorted_pp, 1):
    sev_icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(pp["severity"], "⚪")
    print(f"  {i:2d}. {sev_icon} {pp['text'][:65]:65s} — {pp['author']}")

print(f"\nALL FEATURE REQUESTS")
print("=" * 70)

for i, fr in enumerate(all_feature_requests, 1):
    print(f"  {i:2d}. ✨ {fr['text'][:65]:65s} — {fr['author']}")

---

# Part E — Theme Clustering

## 18. LLM-Based Theme Clustering

Group similar pain points and feature requests into higher-level themes. This is like topic modeling but done by the LLM — it can understand semantic similarity without embeddings.

In [ ]:
CLUSTER_PROMPT = """Below is a list of customer pain points extracted from reviews.
Group them into 4-6 high-level THEMES. Each theme should have a short name
and list which pain points belong to it.

Pain points:
{items}

Return ONLY JSON:
[{{
  "theme": "Theme Name",
  "description": "one sentence about this theme",
  "items": ["pain point 1", "pain point 2"],
  "count": 2,
  "priority": "high" or "medium" or "low"
}}, ...]"""

pp_list = "\n".join(f"- {pp['text']}" for pp in all_pain_points)

cluster_resp = ask(
    CLUSTER_PROMPT.format(items=pp_list),
    temperature=TEMP_EXTRACT,
)
pain_themes = parse_json(cluster_resp)

print("PAIN POINT THEMES")
print("=" * 70)

if pain_themes and isinstance(pain_themes, list):
    for theme in pain_themes:
        prio_icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(
            theme.get("priority", ""), "⚪")
        print(f"\n{prio_icon} {theme.get('theme', '?')} "
              f"({theme.get('count', '?')} issues)")
        print(f"  {theme.get('description', '')}")
        for item in theme.get("items", []):
            print(f"    • {item}")
else:
    print("(parse error)")
    print(cluster_resp[:300])

In [ ]:
# ── Same for feature requests ─────────────────────────
fr_list = "\n".join(f"- {fr['text']}" for fr in all_feature_requests)

fr_cluster_resp = ask(
    CLUSTER_PROMPT.replace("pain points", "feature requests").format(items=fr_list),
    temperature=TEMP_EXTRACT,
)
feature_themes = parse_json(fr_cluster_resp)

print("FEATURE REQUEST THEMES")
print("=" * 70)

if feature_themes and isinstance(feature_themes, list):
    for theme in feature_themes:
        prio_icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(
            theme.get("priority", ""), "⚪")
        print(f"\n{prio_icon} {theme.get('theme', '?')} "
              f"({theme.get('count', '?')} requests)")
        print(f"  {theme.get('description', '')}")
        for item in theme.get("items", []):
            print(f"    • {item}")
else:
    print("(parse error)")
    print(fr_cluster_resp[:300])

---

# Part F — Summary Dashboard

## 19. Executive Summary — LLM Generated

In [ ]:
EXEC_SYSTEM = """You write executive summaries for product teams. Be direct, data-driven,
and actionable. Format as bullet points grouped by section."""

EXEC_PROMPT = """Write an executive summary of customer review analysis for a product team.

DATA:
- Total reviews analyzed: {n_reviews}
- Sentiment: {pos} positive, {neu} neutral, {neg} negative
- Average rating: {avg_rating:.1f}/5
- Top pain points: {pain_points}
- Top feature requests: {feature_requests}
- Pain point themes: {themes}

Write:
1. OVERALL HEALTH (2-3 sentences on general sentiment)
2. TOP 3 ISSUES (ranked by severity and frequency)
3. TOP 3 OPPORTUNITIES (most requested features)
4. RECOMMENDED ACTIONS (3-5 specific next steps for the product team)

Executive summary:"""

top_pp = ", ".join(pp["text"][:40] for pp in sorted_pp[:5])
top_fr = ", ".join(fr["text"][:40] for fr in all_feature_requests[:5])
theme_names = ", ".join(
    t.get("theme", "?") for t in (pain_themes or [])
) if pain_themes else "(none)"

exec_summary = ask(
    EXEC_PROMPT.format(
        n_reviews=len(REVIEWS),
        pos=sent_counts.get("positive", 0),
        neu=sent_counts.get("neutral", 0),
        neg=sent_counts.get("negative", 0),
        avg_rating=avg,
        pain_points=top_pp,
        feature_requests=top_fr,
        themes=theme_names,
    ),
    system=EXEC_SYSTEM,
    temperature=TEMP_SUMMARY,
)

print("EXECUTIVE SUMMARY")
print("=" * 70)
for line in exec_summary.split("\n"):
    print(textwrap.fill(line, width=70))

## 20. Summary Dashboard

In [ ]:
print("╔" + "═" * 68 + "╗")
print("║  CUSTOMER REVIEW ANALYSIS — DASHBOARD                              ║")
print("╚" + "═" * 68 + "╝")

# ── KPI Row ───────────────────────────────────────────
print(f"\n  📊 Reviews: {len(REVIEWS)}    "
      f"⭐ Avg Rating: {avg:.1f}/5    "
      f"🟢 Positive: {sent_counts.get('positive',0)}    "
      f"🔴 Negative: {sent_counts.get('negative',0)}")

# ── Sentiment Bar Chart ────────────────────────────────
print(f"\n{'─'*70}")
print("  SENTIMENT")
for label in ["positive", "neutral", "negative"]:
    c = sent_counts.get(label, 0)
    pct = 100 * c / total
    bar = "█" * int(pct / 2.5)
    icon = {"positive": "🟢", "neutral": "🟡", "negative": "🔴"}[label]
    print(f"    {icon} {label:8s} {bar:25s} {c:2d} ({pct:.0f}%)")

# ── Top Topics ─────────────────────────────────────────
print(f"\n{'─'*70}")
print("  TOP TOPICS")
for topic, count in topic_counts.most_common(8):
    bar = "█" * (count * 3)
    print(f"    {topic:30s} {bar} {count}")

# ── Severity Distribution ──────────────────────────────
print(f"\n{'─'*70}")
print("  ISSUE SEVERITY")
sev_counts = Counter(e["severity"] for e in extraction_results if e["pain_points"])
for sev in ["high", "medium", "low"]:
    c = sev_counts.get(sev, 0)
    icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}[sev]
    print(f"    {icon} {sev:8s} {'█' * (c * 4)} {c}")

# ── Top Pain Points ────────────────────────────────────
print(f"\n{'─'*70}")
print("  TOP 5 PAIN POINTS")
for i, pp in enumerate(sorted_pp[:5], 1):
    sev_icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(pp["severity"], "⚪")
    print(f"    {i}. {sev_icon} {pp['text'][:55]:55s} [{pp['author']}]")

# ── Top Feature Requests ───────────────────────────────
print(f"\n{'─'*70}")
print("  TOP 5 FEATURE REQUESTS")
for i, fr in enumerate(all_feature_requests[:5], 1):
    print(f"    {i}. ✨ {fr['text'][:55]:55s} [{fr['author']}]")

# ── Review Sources ─────────────────────────────────────
print(f"\n{'─'*70}")
print("  REVIEW SOURCES")
source_counts = Counter(r["source"] for r in REVIEWS)
for src, count in source_counts.most_common():
    print(f"    {src:15s} {'█' * (count * 3)} {count}")

print(f"\n{'═'*70}")

---

## 21. Interpretation — What Does This Data Tell Us?

### Sentiment Findings

The sentiment distribution gives us the **overall health** of customer satisfaction. Key things to look for:

- **Positive-to-negative ratio**: A healthy product has > 3:1 positive to negative. Below 2:1 indicates systemic issues.
- **Neutral reviews**: These are the "swing" group — they like the product enough to stay but have reservations. Converting these to positive is the highest-ROI effort.
- **Low-confidence classifications**: Reviews where the LLM is unsure (< 70% confidence) are often **mixed reviews** — positive overall but with specific complaints. These are your gold mine for targeted improvements.

### Topic Insights

The **topic × sentiment cross-tab** is the most actionable view:

- Topics with high mention count AND negative sentiment = **urgent fires** to fix
- Topics with positive sentiment = your **competitive advantages** — protect and promote these
- Topics never mentioned = potential **blind spots** — customers may not know these features exist

### Pain Point Themes

Clustering individual complaints into themes reveals **systemic issues** vs **one-off problems**:

- Themes with multiple pain points across different users = **real product gaps**
- Single-user complaints = may be edge cases or user error
- High-severity themes with paying customers (Enterprise) = **churn risk** — prioritize immediately

### Feature Requests as Product Strategy

Feature requests tell you what customers **expect but don't have**. The clustering reveals:

- Which feature gaps are blocking deals or causing churn
- Where competitor comparisons come up ("Asana does this", "Todoist handles this")
- Whether requests align with your product roadmap or signal a pivot

## 22. Deep Dive — Analyze Specific Reviews

Pick the most insightful reviews for detailed analysis.

In [ ]:
DEEP_DIVE_PROMPT = """Analyze this customer review in detail.

Review by {author} ({rating}/5 on {source}):
"{text}"

Provide:
1. CUSTOMER PROFILE — what type of user is this? (team size, role, use case)
2. KEY SENTIMENT DRIVERS — what specifically makes them happy or unhappy?
3. CHURN RISK — low/medium/high and why
4. ACTION ITEMS — what should the product team do about this review?

Be specific and actionable (not generic)."""

# Pick the most negative and most positive reviews
extreme_reviews = [
    next(r for r in REVIEWS if r["id"] == 16),  # Data loss
    next(r for r in REVIEWS if r["id"] == 8),   # SSO misleading
    next(r for r in REVIEWS if r["id"] == 9),   # Happy switcher
]

print("DEEP DIVE — SELECTED REVIEWS")
print("=" * 70)

for r in extreme_reviews:
    analysis = ask(
        DEEP_DIVE_PROMPT.format(
            author=r["author"], rating=r["rating"],
            source=r["source"], text=r["text"],
        ),
        temperature=TEMP_SUMMARY,
    )
    stars = "★" * r["rating"] + "☆" * (5 - r["rating"])
    print(f"\n{'─'*70}")
    print(f"Review #{r['id']} — {r['author']} {stars}")
    print(f"{'─'*70}")
    for line in analysis.split("\n"):
        print(textwrap.fill(line, width=70))

## 23. Generate Response Drafts

For negative reviews, draft a professional response the support team can send.

In [ ]:
RESPONSE_SYSTEM = """You write professional, empathetic customer support responses.
Rules:
- Acknowledge the specific issue (don't be generic)
- Apologize where appropriate
- Offer a concrete next step
- Keep it under 100 words
- Don't be defensive"""

RESPONSE_PROMPT = """Write a response to this {rating}-star review:

"{text}"

Review by {author} on {source}.

Tone: professional, empathetic, solution-oriented.
Response:"""

negative_reviews = [r for r in REVIEWS if r["rating"] <= 2]

print("RESPONSE DRAFTS — NEGATIVE REVIEWS")
print("=" * 70)

for r in negative_reviews[:4]:
    response = ask(
        RESPONSE_PROMPT.format(
            rating=r["rating"], text=r["text"],
            author=r["author"], source=r["source"],
        ),
        system=RESPONSE_SYSTEM,
        temperature=TEMP_SUMMARY,
    )
    stars = "★" * r["rating"] + "☆" * (5 - r["rating"])
    print(f"\n{'─'*70}")
    print(f"TO: {r['author']} {stars} ({r['source']})")
    print(f"{'─'*70}")
    print(textwrap.fill(response, width=70))

---

## 24. Batch Analysis — Comparison Across Sources

In [ ]:
print("ANALYSIS BY REVIEW SOURCE")
print("=" * 70)

sources = set(r["source"] for r in REVIEWS)

for source in sorted(sources):
    source_reviews = [r for r in REVIEWS if r["source"] == source]
    source_sentiments = [s for s, r in zip(sentiment_results, REVIEWS)
                         if r["source"] == source]
    source_extractions = [e for e, r in zip(extraction_results, REVIEWS)
                          if r["source"] == source]

    avg_r = sum(r["rating"] for r in source_reviews) / len(source_reviews)
    sent_dist = Counter(s["sentiment"] for s in source_sentiments)
    n_pp = sum(len(e["pain_points"]) for e in source_extractions)

    print(f"\n  📍 {source}")
    print(f"     Reviews: {len(source_reviews)}  |  "
          f"Avg rating: {avg_r:.1f}  |  "
          f"Pain points: {n_pp}")
    print(f"     Sentiment: "
          f"🟢 {sent_dist.get('positive',0)} "
          f"🟡 {sent_dist.get('neutral',0)} "
          f"🔴 {sent_dist.get('negative',0)}")

## 25. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **Binary sentiment** (positive/negative only) | Misses the nuance of mixed or neutral reviews | Use 3+ classes; consider a 1-5 score instead |
| **Rating = sentiment** | A 3-star review can be positive with one complaint | Classify sentiment from TEXT, use rating as validation |
| **Generic topic labels** | "Product" or "General" topics aren't actionable | Use a specific taxonomy the product team can act on |
| **Ignoring feature requests in positive reviews** | Happy customers also have wants | Extract from ALL reviews, not just negative ones |
| **No severity ranking** | Treating a typo complaint the same as data loss | Rank issues by severity × frequency |
| **Over-counting via LLM** | LLM may split one issue into 3 pain points | Review extractions manually; deduplicate with clustering |
| **No cross-tabulation** | Looking at sentiment OR topics, never together | Topic × sentiment reveals WHERE the problems are |
| **Skipping the executive summary** | Dumping raw data on stakeholders | Always synthesize into top 3 issues + recommended actions |

## 26. Production Improvement Ideas

1. **Live data pipeline** — connect to App Store Connect, G2, and Trustpilot APIs for real-time ingestion
2. **Trend analysis** — track sentiment and topics over time (weekly/monthly)
3. **Competitor mentions** — detect when customers mention competitors and map to feature gaps
4. **Automated alerts** — trigger Slack notifications when a high-severity issue appears
5. **Response automation** — auto-draft responses for 1-2 star reviews, send to support for approval
6. **Embedding-based clustering** — use vector embeddings for theme detection at scale (1000+ reviews)
7. **Multi-language support** — translate reviews before analysis; report back in English
8. **NPS prediction** — predict Net Promoter Score from review text alone

## 27. Exercises

### Exercise 1: Add Your Own Reviews

In [ ]:
# ── Exercise 1: Add custom reviews and re-run ─────────
# Add your own reviews to the REVIEWS list and re-run the pipeline

# REVIEWS.append({
#     "id": 21, "author": "Your Name", "rating": 3, "source": "Internal",
#     "text": "Paste a real review here..."
# })
# Then re-run from cell 12 onward.

print("Exercise 1: Add your product's real reviews and re-run the pipeline.")
print("Tip: Copy from App Store, G2, or Trustpilot for realistic data.")

### Exercise 2: Aspect-Based Sentiment Analysis

In [ ]:
# ── Exercise 2: Sentiment per ASPECT, not per review ──
# A single review can be positive about "mobile app" but negative about "support"

ASPECT_PROMPT = """Perform aspect-based sentiment analysis on this review.

For each aspect mentioned, classify its sentiment separately.

Review:
"{text}"

Return ONLY JSON:
{{
  "aspects": [
    {{"aspect": "feature name", "sentiment": "positive/negative/neutral", "quote": "relevant quote"}}
  ]
}}"""

# Test on a mixed review
mixed_review = REVIEWS[2]  # Lisa K. — 4 stars, positive + complaint

resp = ask(ASPECT_PROMPT.format(text=mixed_review["text"]), temperature=TEMP_EXTRACT)
parsed = parse_json(resp)

print(f"ASPECT-BASED SENTIMENT — #{mixed_review['id']} {mixed_review['author']}")
print(f"Overall: {'★' * mixed_review['rating']}{'☆' * (5 - mixed_review['rating'])}")
print("=" * 60)

if parsed and "aspects" in parsed:
    for a in parsed["aspects"]:
        icon = {"positive": "🟢", "neutral": "🟡", "negative": "🔴"}.get(
            a.get("sentiment", ""), "⚪")
        print(f"  {icon} {a.get('aspect', '?'):25s} — \"{a.get('quote', '')[:50]}\"")
else:
    print(resp[:200])

print("\n→ Notice: one review can have DIFFERENT sentiments per aspect!")

### Exercise 3: Competitive Intelligence

In [ ]:
# ── Exercise 3: Detect competitor mentions ────────────

COMPETITOR_PROMPT = """Analyze these customer reviews for competitor mentions.

Reviews:
{reviews}

For each competitor mentioned, report:
- competitor name
- context (why they were mentioned)
- whether the comparison is favorable or unfavorable for our product
- review ID

Return ONLY JSON array:
[{{"competitor": "...", "context": "...", "favorable": true/false, "review_id": N}}]"""

reviews_text = "\n\n".join(
    f"[Review #{r['id']}] {r['text']}" for r in REVIEWS
)

comp_resp = ask(
    COMPETITOR_PROMPT.format(reviews=reviews_text),
    temperature=TEMP_EXTRACT,
)
competitors = parse_json(comp_resp)

print("COMPETITIVE INTELLIGENCE")
print("=" * 60)

if competitors and isinstance(competitors, list):
    for c in competitors:
        fav = "✅ Favorable" if c.get("favorable") else "❌ Unfavorable"
        print(f"\n  🏢 {c.get('competitor', '?')} (Review #{c.get('review_id', '?')})")
        print(f"     {fav}")
        print(f"     {c.get('context', '')}")
else:
    print(comp_resp[:300])

### Mini Challenge

1. **Trend simulation:** Assign fake dates to each review (spread over 6 months). Compute monthly sentiment trends. Does sentiment improve or deteriorate? Visualize as a text-based sparkline.

2. **Priority matrix:** Build a 2×2 matrix of pain points: (frequency × severity). "High frequency + high severity" = fix immediately. "Low frequency + high severity" = monitor. Which quadrant has the most items?

3. **Auto-tagging pipeline:** Build a function that takes ANY new review and returns a complete analysis card: sentiment, topics, pain points, feature requests, severity, and a suggested response — all in one LLM call.

4. **Review quality scoring:** Not all reviews are equally useful. Score each review on information density (does it contain specific, actionable feedback vs. vague praise/complaints?). Which reviews should the PM read first?

## 28. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nDataset: {len(REVIEWS)} customer reviews")
print(f"Average rating: {avg:.1f}/5")
print(f"Sources: {', '.join(sorted(sources))}")

print(f"\nSentiment:")
print(f"  🟢 Positive: {sent_counts.get('positive',0)}")
print(f"  🟡 Neutral:  {sent_counts.get('neutral',0)}")
print(f"  🔴 Negative: {sent_counts.get('negative',0)}")

print(f"\nTopics detected: {len(topic_counts)}")
print(f"Pain points extracted: {total_pp}")
print(f"Feature requests extracted: {total_fr}")
print(f"Pain point themes: {len(pain_themes) if pain_themes else 0}")
print(f"Feature request themes: {len(feature_themes) if feature_themes else 0}")

print(f"\nPipeline stages completed:")
print(f"  ✅ Review dataset (20 reviews, 4 sources)")
print(f"  ✅ Sentiment analysis with confidence scoring")
print(f"  ✅ Topic extraction with predefined taxonomy")
print(f"  ✅ Pain point & feature request extraction")
print(f"  ✅ LLM-based theme clustering")
print(f"  ✅ Executive summary & dashboard")
print(f"  ✅ Deep-dive analysis & response drafts")

## 29. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Multi-pass analysis** (sentiment → topics → extraction → clustering) gives richer insights than a single prompt |
| 2 | **Predefined taxonomy** for topics keeps results consistent and aggregatable across reviews |
| 3 | **Severity scoring** makes pain points actionable — not all complaints are equally urgent |
| 4 | **Topic × sentiment cross-tab** is the most actionable view for product teams |
| 5 | **LLM clustering** finds themes without embeddings — good for small datasets (< 100 reviews) |
| 6 | **Response drafts** turn analysis into action — don't just analyze, help the team respond |
| 7 | **Aspect-based sentiment** reveals that one review can be positive about UX but negative about pricing |
| 8 | **The executive summary** is what stakeholders actually read — invest time in making it actionable |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*